# 10 - Alzheimer HF Black-box Shadow Model MIA

Notebook autonome pour tester une attaque MIA de type shadow-model en mode black-box sur la data Alzheimer HF préparée.

Objectifs:
- Charger prepared_alzheimer_hf_transformer.npz
- Entrainer un modele cible tabulaire sur grand dataset
- Introduire un legere overfitting par subset training (sans fuite de donnees)
- Evaluer une attaque shadow-model standard
- Comparer shadow_meta avec les baselines loss et logistic

In [ ]:
import os
import sys
import random
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow.keras import layers, Model

repo_root = os.getcwd()
proto_dir = os.path.join(repo_root, 'transformer_pipeline')
if proto_dir not in sys.path:
    sys.path.append(proto_dir)

import research_protocol as rp
importlib.reload(rp)
evaluate_mia_research_protocol = rp.evaluate_mia_research_protocol
set_seed = rp.set_seed

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

def set_global_determinism(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [ ]:
SEED = 42
ATTACK_SEEDS = [11, 22, 33, 44, 55]

# Target model - adapted for large dataset
TARGET_DROPOUT = 0.00
TARGET_EPOCHS = 100
TARGET_BATCH_SIZE = 16
TARGET_LR = 1e-3
TARGET_MEMBER_FRACTION = 0.50

# Shadow-model attack settings - lighter for large dataset
N_SHADOWS = 25
SHADOW_EPOCHS = 50
SHADOW_BATCH_SIZE = 16
SHADOW_LR = 1e-3
SHADOW_MEMBER_FRACTION = TARGET_MEMBER_FRACTION

ATTACK_MIN_AUC_TARGET = 0.55

set_global_determinism(SEED)

prepared_path = os.path.join('..', 'data_preparation_alzheimer_hf', 'prepared_alzheimer_hf_transformer.npz')
if not os.path.exists(prepared_path):
    raise FileNotFoundError('prepared_alzheimer_hf_transformer.npz introuvable. Executer le notebook de data prep Alzheimer HF.')

b = np.load(prepared_path, allow_pickle=True)
X_train = b['X_train'].astype(np.float32)
X_test = b['X_test'].astype(np.float32)
y_train = b['y_train'].astype(np.int32)
y_test = b['y_test'].astype(np.int32)
X_shadow_raw = b['X_shadow_raw'].astype(np.float32)
y_shadow = b['y_shadow'].astype(np.int32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1]).astype(np.float32)
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1]).astype(np.float32)

# Build target member subset (strict subset of train split only)
rng_target = np.random.default_rng(SEED + 2026)
n_target_members = max(32, int(len(X_train_seq) * TARGET_MEMBER_FRACTION))
target_member_idx = rng_target.choice(len(X_train_seq), size=n_target_members, replace=False)
X_train_target_seq = X_train_seq[target_member_idx]
y_train_target = y_train[target_member_idx]

print(f'Prepared file: {prepared_path}')
print(f'Sizes: train={len(X_train_seq)}, test={len(X_test_seq)}, shadow={len(X_shadow_raw)}')
print(f'Target member subset: {len(X_train_target_seq)} / {len(X_train_seq)} ({TARGET_MEMBER_FRACTION:.0%})')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}, shadow={y_shadow.mean():.4f}')

def build_tabular_mlp(n_features, dropout=0.15, width1=384, width2=256, width3=128, lr=1e-3):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Flatten()(inp)
    x = layers.Dense(width1, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(width2, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(width3, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

In [ ]:
print('=== Target model (overfitting-focused) ===')
set_seed(SEED + 2)
tf.keras.backend.clear_session()

model_attack = build_tabular_mlp(
    n_features=X_train.shape[1],
    dropout=TARGET_DROPOUT,
    width1=384,
    width2=256,
    width3=128,
    lr=TARGET_LR,
)
model_attack.fit(
    X_train_target_seq,
    y_train_target,
    epochs=TARGET_EPOCHS,
    batch_size=TARGET_BATCH_SIZE,
    verbose=0,
)

p_tr_attack_member = model_attack.predict(X_train_target_seq, verbose=0).ravel()
p_tr_attack_full = model_attack.predict(X_train_seq, verbose=0).ravel()
p_te_attack = model_attack.predict(X_test_seq, verbose=0).ravel()

attack_train_member_acc = accuracy_score(y_train_target, (p_tr_attack_member >= 0.5).astype(int))
attack_train_full_acc = accuracy_score(y_train, (p_tr_attack_full >= 0.5).astype(int))
attack_test_acc = accuracy_score(y_test, (p_te_attack >= 0.5).astype(int))

display(pd.DataFrame([{
    'model': 'attack_target',
    'train_acc_member_subset': attack_train_member_acc,
    'train_acc_full_pool': attack_train_full_acc,
    'test_acc': attack_test_acc,
    'gap_member_minus_test': attack_train_member_acc - attack_test_acc,
    'train_auc_member_subset': roc_auc_score(y_train_target, p_tr_attack_member),
    'test_auc': roc_auc_score(y_test, p_te_attack),
}]).round(4))

print(f'✓ Target config: {TARGET_EPOCHS} epochs, batch={TARGET_BATCH_SIZE}, lr={TARGET_LR}')
print(f'✓ Architecture: 384->256->128, dropout={TARGET_DROPOUT}, member_fraction={TARGET_MEMBER_FRACTION}')

In [ ]:
print('=== Shadow-model black-box attack ===')
import inspect

extra_kwargs = {}
sig = inspect.signature(evaluate_mia_research_protocol)
if 'optimize_low_fpr_threshold' in sig.parameters:
    extra_kwargs['optimize_low_fpr_threshold'] = True
if 'max_fpr_target' in sig.parameters:
    extra_kwargs['max_fpr_target'] = 0.05
if 'shadow_member_fraction' in sig.parameters:
    extra_kwargs['shadow_member_fraction'] = SHADOW_MEMBER_FRACTION

baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
    target_model=model_attack,
    X_train_seq=X_train_target_seq,
    y_train=y_train_target,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_tabular_mlp(n_features=nf, dropout=0.15, width1=384, width2=256, width3=128, lr=SHADOW_LR),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH_SIZE,
    **extra_kwargs,
)
display(baseline_summary.round(4))

quick_view = baseline_summary[['attack', 'auc_mean', 'auc_std', 'tpr_at_1pct_fpr_mean', 'tpr_at_5pct_fpr_mean']].copy()
display(quick_view.round(4))

shadow_auc = float(baseline_summary.loc[baseline_summary['attack'] == 'shadow_meta', 'auc_mean'].iloc[0])
shadow_tpr_5 = float(baseline_summary.loc[baseline_summary['attack'] == 'shadow_meta', 'tpr_at_5pct_fpr_mean'].iloc[0])

if shadow_auc < ATTACK_MIN_AUC_TARGET:
    print(f'⚠️  WARNING: shadow_meta={shadow_auc:.4f} (below target {ATTACK_MIN_AUC_TARGET:.2f})')
else:
    print(f'✓ GOOD: shadow_meta AUC={shadow_auc:.4f} | TPR@5% FPR={shadow_tpr_5:.4f}')

print(f'✓ Attack config: {N_SHADOWS} shadows × {SHADOW_EPOCHS} epochs, batch={SHADOW_BATCH_SIZE}, lr={SHADOW_LR}, member_fraction={SHADOW_MEMBER_FRACTION}')